In [1]:
import pandas as pd
import numpy as np

In [2]:
from transformers import pipeline
from google.colab import files, drive
import torch
from tqdm.auto import tqdm
from datasets import Dataset
from transformers.pipelines.pt_utils import KeyDataset

In [3]:
df = pd.read_excel('/content/Merged_Final_Dataset.xlsx',sheet_name='Sheet1',usecols = [' CompanyNumber','Google_Review_Texts'])

In [4]:
df.head()

,CompanyNumber,Google_Review_Texts
0,00020100,I’ve been to multiple meetings and events at t...
1,00024166,I have been a member in here for many years no...
2,00024509,"Lovely club, known for their great beer and di..."
3,00066211,We booked this venue for our wedding. It was a...
4,00079244,This is a fantastic venue. I had a party here ...


In [5]:
len(df)

24748

In [6]:
df = df.dropna(subset = ['Google_Review_Texts'])

In [7]:
len(df)

24622

In [8]:
df['Google_Review_Texts'] = df['Google_Review_Texts'].str.split('|')

In [9]:
df = df.explode('Google_Review_Texts')

In [10]:
df.head(10)

,CompanyNumber,Google_Review_Texts
0,00020100,I’ve been to multiple meetings and events at t...
0,00020100,Being a member of this charming club is a joy...
0,00020100,"Beautiful building, awful people inside. Very..."
0,00020100,Staff are courteous and professional. The who...
0,00020100,A little oasis in a modern metropolis ... Bu...
1,00024166,I have been a member in here for many years no...
1,00024166,Stopped by for a meal as I had heard the new ...
1,00024166,Absolutely disgusted with this place especial...
1,00024166,Great club..everyone is friendly food is real...
1,00024166,Go here for my slimming world meeting on a Tu...


In [11]:
df['Google_Review_Texts'] = df['Google_Review_Texts'].astype(str).str.strip()

In [12]:
df = df.dropna(subset = ['Google_Review_Texts'])

In [13]:
len(df)

121093

In [14]:
df = df[df['Google_Review_Texts'] != ""]

In [15]:
df = df.drop_duplicates(subset=[' CompanyNumber', 'Google_Review_Texts'])

In [16]:
len(df)

119932

In [17]:
device = 0 if torch.cuda.is_available() else -1
print(f"Loading model on {'GPU' if device == 0 else 'CPU'}...")

Loading model on GPU...


In [18]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device,torch_dtype=torch.float16)
print(f"  VRAM used after model load : {torch.cuda.memory_allocated() / 1e9:.2f} GB\n")
categories = [
    "Food and Beverage Quality",
    "Customer Service",
    "Delivery & Order Accuracy",
    "Value for Money",
    "Ambiance & Atmosphere",
    "Cleanliness & Hygiene",
    "Facilities & Amenities"
]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

  VRAM used after model load : 0.81 GB



In [19]:
texts = df["Google_Review_Texts"].fillna("no review").str.strip().replace("", "no review").tolist()
hf_dataset = Dataset.from_dict({"text": texts})

In [20]:
output_path = '/content/Scored_Reviews_Saved.csv'

In [21]:
print("Scoring reviews and saving progress to Google Drive...")
batch_size = 64 # Safe limit to prevent RAM crashes
save_interval = 10000
parsed_results = []
last_save_idx = 0
total = len(texts)
# Create a manual progress bar
with tqdm(total=total, desc="Processing") as pbar:
    for i, result in enumerate(
        classifier(
            KeyDataset(hf_dataset, "text"),
            candidate_labels=categories,
            multi_label=True,
            batch_size=batch_size,
        )
    ):
        scores = {
            label: score
            for label, score in zip(result["labels"], result["scores"])
        }
        parsed_results.append(scores)
        pbar.update(1)

        current_idx = i + 1
        is_last     = current_idx == total
        should_save = (current_idx - last_save_idx) >= save_interval or is_last

        if should_save:
            chunk_scores    = pd.DataFrame(parsed_results[last_save_idx:current_idx])
            chunk_original  = df.iloc[last_save_idx:current_idx].reset_index(drop=True)
            chunk_df        = pd.concat([chunk_original, chunk_scores], axis=1)

            if last_save_idx == 0:
                chunk_df.to_csv(output_path, index=False, mode="w")
            else:
                chunk_df.to_csv(output_path, index=False, mode="a", header=False)

            last_save_idx = current_idx

            if torch.cuda.is_available():
                print(
                    f"  Saved {current_idx:,}/{total:,} rows | "
                    f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB"
                )

print(f"\nDone! Scored file saved to: {output_path}")

Scoring reviews and saving progress to Google Drive...


Processing:   0%|          | 0/119932 [00:00<?, ?it/s]

  Saved 10,000/119,932 rows | VRAM: 0.82 GB
  Saved 20,000/119,932 rows | VRAM: 0.82 GB
  Saved 30,000/119,932 rows | VRAM: 0.82 GB
  Saved 40,000/119,932 rows | VRAM: 0.82 GB
  Saved 50,000/119,932 rows | VRAM: 0.82 GB
  Saved 60,000/119,932 rows | VRAM: 0.82 GB
  Saved 70,000/119,932 rows | VRAM: 0.82 GB
  Saved 80,000/119,932 rows | VRAM: 0.82 GB
  Saved 90,000/119,932 rows | VRAM: 0.82 GB
  Saved 100,000/119,932 rows | VRAM: 0.82 GB
  Saved 110,000/119,932 rows | VRAM: 0.82 GB
  Saved 119,932/119,932 rows | VRAM: 0.82 GB

Done! Scored file saved to: /content/Scored_Reviews_Saved.csv


In [22]:
final_df = pd.read_csv('/content/Scored_Reviews_Saved.csv')

In [23]:
final_df.head(10)

,CompanyNumber,Google_Review_Texts,Value for Money,Ambiance & Atmosphere,Facilities & Amenities,Customer Service,Delivery & Order Accuracy,Food and Beverage Quality,Cleanliness & Hygiene
0,00020100,I’ve been to multiple meetings and events at t...,0.990487,0.933945,0.914224,0.735071,0.689619,0.470615,0.326057
1,00020100,Being a member of this charming club is a joy ...,0.971476,0.929856,0.440229,0.144480,0.397237,0.102193,0.311532
2,00020100,"Beautiful building, awful people inside. Very ...",0.004455,0.699280,0.193604,0.066100,0.213800,0.001694,0.121113
3,00020100,Staff are courteous and professional. The whol...,0.995558,0.993867,0.759493,0.955402,0.804560,0.967686,0.567218
4,00020100,A little oasis in a modern metropolis ... Bus...,0.959164,0.848784,0.253272,0.090253,0.242963,0.803097,0.030444
5,00024166,I have been a member in here for many years no...,0.947908,0.949189,0.813609,0.467575,0.773559,0.810854,0.575349
6,00024166,Stopped by for a meal as I had heard the new k...,0.994969,0.969785,0.714105,0.822359,0.536890,0.988039,0.072900
7,00024166,Absolutely disgusted with this place especiall...,0.029146,0.511717,0.531939,0.654895,0.628977,0.536313,0.625438
8,00024166,Great club..everyone is friendly food is reall...,0.959649,0.893681,0.781550,0.211376,0.097369,0.955734,0.002884
9,00024166,Go here for my slimming world meeting on a Tue...,0.921393,0.525856,0.654784,0.062495,0.016834,0.000714,0.905761


In [24]:
len(final_df)

119932

In [25]:
len(df)

119932

In [26]:
df = df.reset_index()

In [27]:
df = df.drop(['index'],axis=1)

In [28]:
df.head(10)

,CompanyNumber,Google_Review_Texts
0,00020100,I’ve been to multiple meetings and events at t...
1,00020100,Being a member of this charming club is a joy ...
2,00020100,"Beautiful building, awful people inside. Very ..."
3,00020100,Staff are courteous and professional. The whol...
4,00020100,A little oasis in a modern metropolis ... Bus...
5,00024166,I have been a member in here for many years no...
6,00024166,Stopped by for a meal as I had heard the new k...
7,00024166,Absolutely disgusted with this place especiall...
8,00024166,Great club..everyone is friendly food is reall...
9,00024166,Go here for my slimming world meeting on a Tue...


In [29]:
merged_df = pd.merge(
    df,
    final_df,
    on=[' CompanyNumber', 'Google_Review_Texts'],
    how='inner'
)

In [30]:
merged_df.head()

,CompanyNumber,Google_Review_Texts,Value for Money,Ambiance & Atmosphere,Facilities & Amenities,Customer Service,Delivery & Order Accuracy,Food and Beverage Quality,Cleanliness & Hygiene
0,00020100,I’ve been to multiple meetings and events at t...,0.990487,0.933945,0.914224,0.735071,0.689619,0.470615,0.326057
1,00020100,Being a member of this charming club is a joy ...,0.971476,0.929856,0.440229,0.144480,0.397237,0.102193,0.311532
2,00020100,"Beautiful building, awful people inside. Very ...",0.004455,0.699280,0.193604,0.066100,0.213800,0.001694,0.121113
3,00020100,Staff are courteous and professional. The whol...,0.995558,0.993867,0.759493,0.955402,0.804560,0.967686,0.567218
4,00020100,A little oasis in a modern metropolis ... Bus...,0.959164,0.848784,0.253272,0.090253,0.242963,0.803097,0.030444


In [31]:
len(merged_df)

119932

In [32]:
# Find combinations that appear more than once in your original data
duplicates = df[df.duplicated(subset=[' CompanyNumber', 'Google_Review_Texts'], keep=False)]
print(f"Rows with duplicate keys in df: {len(duplicates)}")

# See the specific repeated reviews
print(duplicates.groupby([' CompanyNumber', 'Google_Review_Texts']).size().reset_index(name='count').sort_values('count', ascending=False))

Rows with duplicate keys in df: 0
Empty DataFrame
Columns: [ CompanyNumber, Google_Review_Texts, count]
Index: []
